In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

### Label construction and candidate generation for rernker training pairs

In [2]:
DATA_ROOT = Path("../../data")
ARTIFACT_DIR = Path("../../ml/artifacts/content_based_anime_v1")

INTERACTIONS_PATH = DATA_ROOT / "processed" / "interactions_clean.parquet"
ITEM_FEATURES_PARQUET = DATA_ROOT / "processed" / "item_features.parquet"
ITEM_FEATURES_CSV = DATA_ROOT / "processed" / "item_features.csv"
METADATA_CSV = ARTIFACT_DIR / "anime_metadata.csv"
METADATA_PARQUET = ARTIFACT_DIR / "anime_metadata.parquet"
EMBEDDINGS_PATH = ARTIFACT_DIR / "synopsis_embeddings.npy"
OUTPUT_PATH = DATA_ROOT / "processed" / "reranker_pairs.parquet"

RNG = np.random.default_rng(42)

# Sampling policy — increased for richer LambdaRank signal
HARD_NEG_RANGE = (5, 8)       # user's low-rated items (was 3-5)
STAGE1_NEG_RANGE = (5, 8)     # NEW: embedding-similar but low-rated or unseen
EMB_NEG_RANGE = (3, 5)        # top embedding-similar, unseen (was 2-3)
POP_NEG_RANGE = (2, 4)        # popular unseen items (was 1-2)

# Target ~20-30 negatives per positive
MIN_NEGATIVES_PER_POSITIVE = 15

# Memory guards
MAX_POSITIVES_PER_USER = 30
MAX_USERS_FOR_TRAINING = 20000
MAX_TOTAL_PAIRS = 3_000_000

In [3]:
# ------------------------------------------------------------
# 1) Load interactions and construct labels
interactions = pd.read_parquet(INTERACTIONS_PATH, columns=["user_id", "anime_id", "rating"]).copy()
interactions["user_id"] = pd.to_numeric(interactions["user_id"], errors="coerce")
interactions["anime_id"] = pd.to_numeric(interactions["anime_id"], errors="coerce")
interactions["rating"] = pd.to_numeric(interactions["rating"], errors="coerce")
interactions = interactions.dropna(subset=["user_id", "anime_id", "rating"]).copy()
interactions["user_id"] = interactions["user_id"].astype("int64")
interactions["anime_id"] = interactions["anime_id"].astype("int64")
interactions["rating"] = interactions["rating"].astype("int8")

# Drop ambiguous unrated rows (rating == 0)
interactions = interactions[interactions["rating"] != 0].copy()

# Binary label mapping: >=7 -> 1, 1-6 -> 0
interactions = interactions[(interactions["rating"] >= 1) & (interactions["rating"] <= 10)].copy()
interactions["label"] = (interactions["rating"] >= 7).astype("int8")

# Stratified user sampling: sample from activity-level bins for diversity
user_activity = interactions["user_id"].value_counts()
available_users = len(user_activity)

if available_users > MAX_USERS_FOR_TRAINING:
    N_STRATA = min(10, int(user_activity.nunique()))
    if N_STRATA > 1:
        activity_bins = pd.qcut(user_activity, q=N_STRATA, labels=False, duplicates="drop")
        unique_bins = sorted(activity_bins.dropna().unique())
        users_per_bin = MAX_USERS_FOR_TRAINING // len(unique_bins)
        selected_users = []
        remaining_slots = MAX_USERS_FOR_TRAINING

        for i, bin_id in enumerate(unique_bins):
            bin_members = user_activity.index[activity_bins == bin_id].to_numpy()
            bins_left = len(unique_bins) - i
            take = min(len(bin_members), max(1, remaining_slots // bins_left))
            chosen = RNG.choice(bin_members, size=take, replace=False)
            selected_users.extend(chosen.tolist())
            remaining_slots -= take

        # If slightly under target due to rounding, fill from remaining users
        if len(selected_users) < MAX_USERS_FOR_TRAINING:
            already = set(selected_users)
            extra = [u for u in user_activity.index if u not in already][
                : MAX_USERS_FOR_TRAINING - len(selected_users)
            ]
            selected_users.extend(extra)

        selected_users = set(selected_users)
    else:
        selected_users = set(user_activity.index[:MAX_USERS_FOR_TRAINING].tolist())

    interactions = interactions[interactions["user_id"].isin(selected_users)].copy()
    del selected_users

del user_activity
interactions = interactions.reset_index(drop=True)

positives = interactions[interactions["label"] == 1].copy()
negatives = interactions[interactions["label"] == 0].copy()

print(f"Users kept for training pair generation: {interactions['user_id'].nunique():,}")
print(f"Interactions after filtering: {len(interactions):,}")
print("Label distribution:")
print(interactions["label"].value_counts().sort_index())

Users kept for training pair generation: 20,000
Interactions after filtering: 3,356,304
Label distribution:
label
0     742598
1    2613706
Name: count, dtype: int64


In [4]:
# ------------------------------------------------------------
# 2) Load item features for popularity negatives
if ITEM_FEATURES_CSV.exists():
    item_features = pd.read_csv(ITEM_FEATURES_CSV)
elif ITEM_FEATURES_PARQUET.exists():
    item_features = pd.read_parquet(ITEM_FEATURES_PARQUET)
else:
    raise FileNotFoundError("Missing item_features file (csv or parquet)")

item_features["anime_id"] = pd.to_numeric(item_features["anime_id"], errors="coerce")
item_features = item_features.dropna(subset=["anime_id"]).copy()
item_features["anime_id"] = item_features["anime_id"].astype("int64")

if "popularity_rank" not in item_features.columns:
    raise ValueError("item_features must include popularity_rank")

item_features["popularity_rank"] = pd.to_numeric(item_features["popularity_rank"], errors="coerce")
item_features = item_features.dropna(subset=["popularity_rank"]).copy()

top_popular_ids = (
    item_features.sort_values("popularity_rank", ascending=True)["anime_id"]
    .drop_duplicates()
    .head(500)
    .tolist()
)

In [5]:
# ------------------------------------------------------------
# 3) Load embeddings + metadata mapping for embedding & stage-1 negatives
if METADATA_CSV.exists():
    metadata = pd.read_csv(METADATA_CSV)
elif METADATA_PARQUET.exists():
    metadata = pd.read_parquet(METADATA_PARQUET)
else:
    raise FileNotFoundError("Missing anime metadata file (csv or parquet)")

if "anime_id" not in metadata.columns or "anime_index" not in metadata.columns:
    raise ValueError("Metadata must contain anime_id and anime_index for embedding negatives")

metadata["anime_id"] = pd.to_numeric(metadata["anime_id"], errors="coerce")
metadata["anime_index"] = pd.to_numeric(metadata["anime_index"], errors="coerce")
metadata = metadata.dropna(subset=["anime_id", "anime_index"]).copy()
metadata["anime_id"] = metadata["anime_id"].astype("int64")
metadata["anime_index"] = metadata["anime_index"].astype("int64")

embeddings = np.load(EMBEDDINGS_PATH)
if embeddings.ndim != 2:
    raise ValueError(f"Expected 2D embeddings, got shape={embeddings.shape}")

anime_to_index = dict(zip(metadata["anime_id"].tolist(), metadata["anime_index"].tolist()))
index_to_anime = dict(zip(metadata["anime_index"].tolist(), metadata["anime_id"].tolist()))


def embedding_negatives_for_positive(
    pos_anime_id: int, user_seen: set[int], n_take: int
) -> list[int]:
    """Top embedding-similar items that the user has NOT interacted with."""
    if n_take <= 0:
        return []
    if pos_anime_id not in anime_to_index:
        return []

    q_idx = anime_to_index[pos_anime_id]
    if q_idx < 0 or q_idx >= embeddings.shape[0]:
        return []

    sims = embeddings @ embeddings[q_idx]
    sims[q_idx] = -1.0

    order = np.argsort(-sims)
    chosen = []
    for idx in order:
        anime_id = index_to_anime.get(int(idx))
        if anime_id is None:
            continue
        if anime_id in user_seen:
            continue
        chosen.append(int(anime_id))
        if len(chosen) >= n_take:
            break
    return chosen


def stage1_negatives_for_query(
    query_anime_id: int,
    user_neg_set: set[int],
    user_seen: set[int],
    used_negatives: set[int],
    n_take: int,
    top_k: int = 200,
) -> tuple[list[int], list[int]]:
    """Stage-1-aligned negatives: embedding-similar items that mimic inference candidates.

    Returns two lists:
      - stage1_hard: items from top-K by embedding similarity that the user rated LOW
      - stage1_unseen: items from top-K that the user hasn't interacted with
    These are ordered by preference: hard first (stronger signal), then unseen.
    """
    if n_take <= 0:
        return [], []
    if query_anime_id not in anime_to_index:
        return [], []

    q_idx = anime_to_index[query_anime_id]
    if q_idx < 0 or q_idx >= embeddings.shape[0]:
        return [], []

    sims = embeddings @ embeddings[q_idx]
    sims[q_idx] = -1.0
    top_indices = np.argsort(-sims)[: top_k]

    # Divide into hard (user rated low) and unseen
    stage1_hard = []
    stage1_unseen = []
    for idx in top_indices:
        anime_id = index_to_anime.get(int(idx))
        if anime_id is None or anime_id in used_negatives or anime_id == query_anime_id:
            continue
        if anime_id in user_neg_set:
            stage1_hard.append(int(anime_id))
        elif anime_id not in user_seen:
            stage1_unseen.append(int(anime_id))

    return stage1_hard[:n_take], stage1_unseen[:n_take]


In [6]:
# ------------------------------------------------------------
# 4) Prepare per-user histories and generate training pairs
pos_by_user = positives.groupby("user_id")["anime_id"].apply(list).to_dict()
neg_by_user = negatives.groupby("user_id")["anime_id"].apply(list).to_dict()
seen_by_user = interactions.groupby("user_id")["anime_id"].apply(set).to_dict()

user_ids = sorted(set(pos_by_user.keys()))
records = []
stop_early = False

for user_id in user_ids:
    if stop_early:
        break

    user_pos = list(set(int(x) for x in pos_by_user.get(user_id, [])))
    user_neg = neg_by_user.get(user_id, [])
    user_seen = seen_by_user.get(user_id, set())

    # Need at least 2 liked items to build (query, positive_candidate) pairs
    if len(user_pos) < 2:
        continue

    if len(user_pos) > MAX_POSITIVES_PER_USER:
        user_pos = RNG.choice(
            np.array(user_pos, dtype=np.int64), size=MAX_POSITIVES_PER_USER, replace=False
        ).tolist()

    user_neg_list = list(set(int(x) for x in user_neg))
    user_neg_set = set(user_neg_list)

    for query_anime_id in user_pos:
        if len(records) >= MAX_TOTAL_PAIRS:
            stop_early = True
            break

        query_anime_id = int(query_anime_id)
        pos_pool = [aid for aid in user_pos if aid != query_anime_id]
        if not pos_pool:
            continue

        # Positive: another liked item from user history
        positive_candidate = int(
            RNG.choice(np.array(pos_pool, dtype=np.int64), size=1, replace=False)[0]
        )
        records.append(
            {
                "user_id": int(user_id),
                "query_anime_id": query_anime_id,
                "candidate_anime_id": positive_candidate,
                "label": 1,
                "candidate_source": "positive_history",
            }
        )

        used_negatives = {positive_candidate}
        neg_count = 0

        # --- 1) Hard negatives: user's low-rated items ---
        hard_n = int(RNG.integers(HARD_NEG_RANGE[0], HARD_NEG_RANGE[1] + 1))
        hard_pool = [
            aid for aid in user_neg_list
            if aid != query_anime_id and aid not in used_negatives
        ]
        if hard_pool:
            hard_take = min(hard_n, len(hard_pool))
            hard_samples = RNG.choice(
                np.array(hard_pool, dtype=np.int64), size=hard_take, replace=False
            ).tolist()
            for neg_id in hard_samples:
                if len(records) >= MAX_TOTAL_PAIRS:
                    stop_early = True
                    break
                neg_id = int(neg_id)
                used_negatives.add(neg_id)
                neg_count += 1
                records.append(
                    {
                        "user_id": int(user_id),
                        "query_anime_id": query_anime_id,
                        "candidate_anime_id": neg_id,
                        "label": 0,
                        "candidate_source": "hard_negative",
                    }
                )
        if stop_early:
            break

        # --- 2) Stage-1-aligned negatives: embedding-similar items that match
        #     inference distribution (low-rated or unseen but in top-200 by sim) ---
        s1_n = int(RNG.integers(STAGE1_NEG_RANGE[0], STAGE1_NEG_RANGE[1] + 1))
        s1_hard, s1_unseen = stage1_negatives_for_query(
            query_anime_id=query_anime_id,
            user_neg_set=user_neg_set,
            user_seen=user_seen,
            used_negatives=used_negatives,
            n_take=s1_n,
        )
        # Prefer stage1_hard (user rated low + embedding-similar = strongest signal)
        s1_combined = s1_hard + s1_unseen
        s1_take = min(s1_n, len(s1_combined))
        for neg_id in s1_combined[:s1_take]:
            if len(records) >= MAX_TOTAL_PAIRS:
                stop_early = True
                break
            neg_id = int(neg_id)
            used_negatives.add(neg_id)
            neg_count += 1
            records.append(
                {
                    "user_id": int(user_id),
                    "query_anime_id": query_anime_id,
                    "candidate_anime_id": neg_id,
                    "label": 0,
                    "candidate_source": "stage1_negative",
                }
            )
        if stop_early:
            break

        # --- 3) Embedding negatives: top-similar unseen items ---
        emb_n = int(RNG.integers(EMB_NEG_RANGE[0], EMB_NEG_RANGE[1] + 1))
        emb_samples = embedding_negatives_for_positive(
            pos_anime_id=query_anime_id,
            user_seen=user_seen.union(used_negatives),
            n_take=emb_n,
        )
        for neg_id in emb_samples:
            if len(records) >= MAX_TOTAL_PAIRS:
                stop_early = True
                break
            neg_id = int(neg_id)
            used_negatives.add(neg_id)
            neg_count += 1
            records.append(
                {
                    "user_id": int(user_id),
                    "query_anime_id": query_anime_id,
                    "candidate_anime_id": neg_id,
                    "label": 0,
                    "candidate_source": "embedding_negative",
                }
            )
        if stop_early:
            break

        # --- 4) Popularity negatives: random popular items user hasn't seen ---
        pop_n = int(RNG.integers(POP_NEG_RANGE[0], POP_NEG_RANGE[1] + 1))
        pop_pool = [
            aid
            for aid in top_popular_ids
            if aid != query_anime_id and aid not in user_seen and aid not in used_negatives
        ]
        if pop_pool:
            pop_take = min(pop_n, len(pop_pool))
            pop_samples = RNG.choice(
                np.array(pop_pool, dtype=np.int64), size=pop_take, replace=False
            ).tolist()
            for neg_id in pop_samples:
                if len(records) >= MAX_TOTAL_PAIRS:
                    stop_early = True
                    break
                neg_id = int(neg_id)
                used_negatives.add(neg_id)
                neg_count += 1
                records.append(
                    {
                        "user_id": int(user_id),
                        "query_anime_id": query_anime_id,
                        "candidate_anime_id": neg_id,
                        "label": 0,
                        "candidate_source": "popularity_negative",
                    }
                )

        # --- 5) Adaptive fill: if we have fewer negatives than target, add more
        #     stage-1-aligned unseen items to reach MIN_NEGATIVES_PER_POSITIVE ---
        if neg_count < MIN_NEGATIVES_PER_POSITIVE and not stop_early:
            deficit = MIN_NEGATIVES_PER_POSITIVE - neg_count
            _, extra_unseen = stage1_negatives_for_query(
                query_anime_id=query_anime_id,
                user_neg_set=set(),  # skip hard, already collected
                user_seen=user_seen,
                used_negatives=used_negatives,
                n_take=deficit,
                top_k=500,  # wider pool for fill
            )
            for neg_id in extra_unseen[:deficit]:
                if len(records) >= MAX_TOTAL_PAIRS:
                    stop_early = True
                    break
                neg_id = int(neg_id)
                used_negatives.add(neg_id)
                neg_count += 1
                records.append(
                    {
                        "user_id": int(user_id),
                        "query_anime_id": query_anime_id,
                        "candidate_anime_id": neg_id,
                        "label": 0,
                        "candidate_source": "stage1_fill_negative",
                    }
                )

reranker_pairs = pd.DataFrame(records)
del records

reranker_pairs = reranker_pairs.drop_duplicates(
    subset=["user_id", "query_anime_id", "candidate_anime_id", "label", "candidate_source"]
).reset_index(drop=True)
reranker_pairs["user_id"] = reranker_pairs["user_id"].astype("int64")
reranker_pairs["query_anime_id"] = reranker_pairs["query_anime_id"].astype("int64")
reranker_pairs["candidate_anime_id"] = reranker_pairs["candidate_anime_id"].astype("int64")
reranker_pairs["label"] = reranker_pairs["label"].astype("int8")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
reranker_pairs.to_parquet(OUTPUT_PATH, index=False)

print(f"Generated pairs: {len(reranker_pairs):,}")
print(f"Stopped early due to pair cap: {stop_early}")
print("Source distribution:")
print(reranker_pairs["candidate_source"].value_counts())
print("Label distribution:")
print(reranker_pairs["label"].value_counts().sort_index())
avg_neg = reranker_pairs[reranker_pairs["label"] == 0].groupby(
    ["user_id", "query_anime_id"]
).size().mean()
print(f"Avg negatives per query: {avg_neg:.1f}")
print(f"Saved: {OUTPUT_PATH}")
reranker_pairs.head()

Generated pairs: 3,000,000
Stopped early due to pair cap: True
Source distribution:
candidate_source
stage1_negative         977404
hard_negative           792474
embedding_negative      601312
popularity_negative     451248
positive_history        150324
stage1_fill_negative     27238
Name: count, dtype: int64
Label distribution:
label
0    2849676
1     150324
Name: count, dtype: int64
Avg negatives per query: 19.0
Saved: ../../data/processed/reranker_pairs.parquet


,user_id,query_anime_id,candidate_anime_id,label,candidate_source
0,30,1535,16592,1,positive_history
1,30,1535,27899,0,hard_negative
2,30,1535,22319,0,hard_negative
3,30,1535,6707,0,hard_negative
4,30,1535,1818,0,hard_negative


In [7]:
# 2.3 Feature vector construction (chunked pipeline)

RERANKER_FEATURES_DIR = DATA_ROOT / "processed" / "reranker_features_parts"
PAIR_BATCH_SIZE = 100_000

def _parse_genres_to_set(value):
    if pd.isna(value):
        return set()
    text = str(value).strip()
    if not text:
        return set()
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = json.loads(text.replace("'", '"'))
            if isinstance(parsed, list):
                return {str(x).strip().lower() for x in parsed if str(x).strip()}
        except Exception:
            pass
    return {part.strip().lower() for part in text.split(",") if part.strip()}

def _jaccard(a, b):
    if not a and not b:
        return 0.0
    union = a.union(b)
    if not union:
        return 0.0
    return len(a.intersection(b)) / len(union)

def _safe_json_to_dict(text):
    if pd.isna(text):
        return {}
    if isinstance(text, dict):
        return {str(k): float(v) for k, v in text.items()}
    value = str(text).strip()
    if not value:
        return {}
    try:
        parsed = json.loads(value)
        if isinstance(parsed, dict):
            return {str(k): float(v) for k, v in parsed.items()}
    except Exception:
        pass
    return {}

def iter_pair_batches(batch_size=PAIR_BATCH_SIZE):
    cols = ["user_id", "query_anime_id", "candidate_anime_id", "label", "candidate_source"]
    if "reranker_pairs" in globals() and isinstance(reranker_pairs, pd.DataFrame):
        base = reranker_pairs[cols].copy()
        for start in range(0, len(base), batch_size):
            yield base.iloc[start : start + batch_size].copy()
        return

    try:
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(OUTPUT_PATH)
        for batch in pf.iter_batches(batch_size=batch_size, columns=cols):
            yield batch.to_pandas()
        return
    except Exception:
        pass

    base = pd.read_parquet(OUTPUT_PATH, columns=cols)
    for start in range(0, len(base), batch_size):
        yield base.iloc[start : start + batch_size].copy()

In [8]:
# 2.3.a Load static lookup tables
import re

if ITEM_FEATURES_CSV.exists():
    item_features = pd.read_csv(ITEM_FEATURES_CSV)
elif ITEM_FEATURES_PARQUET.exists():
    item_features = pd.read_parquet(ITEM_FEATURES_PARQUET)
else:
    raise FileNotFoundError("Missing item_features file (csv or parquet)")

item_features["anime_id"] = pd.to_numeric(item_features["anime_id"], errors="coerce")
item_features = item_features.dropna(subset=["anime_id"]).copy()
item_features["anime_id"] = item_features["anime_id"].astype("int64")
item_features = item_features.set_index("anime_id", drop=False)

candidate_avg_rating_map = item_features["avg_user_rating"].to_dict()
candidate_rating_count_map = item_features["rating_count"].to_dict()
candidate_positive_ratio_map = item_features["positive_ratio"].to_dict()
candidate_bayesian_norm_map = item_features["bayesian_score_norm"].to_dict()
candidate_popularity_rank_map = item_features["popularity_rank"].to_dict()

# rating_std for candidate_rating_cv
candidate_rating_std_map = item_features["rating_std"].to_dict() if "rating_std" in item_features.columns else {}

# Popularity percentile: rank / total items (lower rank = more popular → higher percentile)
n_items = len(item_features)
candidate_popularity_percentile_map = {
    aid: 1.0 - (rank / n_items) for aid, rank in candidate_popularity_rank_map.items()
} if n_items > 0 else {}

catalog = pd.read_parquet(
    DATA_ROOT / "processed" / "anime.parquet",
    columns=["anime_id", "Name", "Genres", "Type", "Studios", "Source", "Score"],
)
catalog["anime_id"] = pd.to_numeric(catalog["anime_id"], errors="coerce")
catalog = catalog.dropna(subset=["anime_id"]).copy()
catalog["anime_id"] = catalog["anime_id"].astype("int64")
catalog["Name"] = catalog["Name"].fillna("").astype(str).str.strip()
catalog["Type"] = catalog["Type"].fillna("").astype(str).str.strip().str.lower()
catalog["Studios"] = catalog["Studios"].fillna("").astype(str).str.strip().str.lower()
catalog["Source"] = catalog["Source"].fillna("").astype(str).str.strip().str.lower()
catalog["Score"] = pd.to_numeric(catalog["Score"], errors="coerce")
catalog = catalog.set_index("anime_id", drop=False)

genre_map = catalog["Genres"].to_dict()
type_map = catalog["Type"].to_dict()
studio_map = catalog["Studios"].to_dict()
source_map = catalog["Source"].to_dict()
score_map = catalog["Score"].to_dict()
name_map = catalog["Name"].to_dict()

# Title tokenizer for title_token_overlap
_TOKEN_RE = re.compile(r"[a-z0-9]+")
def _tokenize_title(title: str) -> set[str]:
    return set(_TOKEN_RE.findall(title.lower()))

title_tokens_map = {aid: _tokenize_title(name) for aid, name in name_map.items()}

if METADATA_CSV.exists():
    metadata = pd.read_csv(METADATA_CSV)
elif METADATA_PARQUET.exists():
    metadata = pd.read_parquet(METADATA_PARQUET)
else:
    raise FileNotFoundError("Missing anime metadata for embedding index mapping")

metadata["anime_id"] = pd.to_numeric(metadata["anime_id"], errors="coerce")
metadata["anime_index"] = pd.to_numeric(metadata["anime_index"], errors="coerce")
metadata = metadata.dropna(subset=["anime_id", "anime_index"]).copy()
metadata["anime_id"] = metadata["anime_id"].astype("int64")
metadata["anime_index"] = metadata["anime_index"].astype("int64")
anime_to_index = dict(zip(metadata["anime_id"].tolist(), metadata["anime_index"].tolist()))
embeddings = np.load(EMBEDDINGS_PATH).astype(np.float32)

# Stage 1 feature weights (from recommendation_v1.py) for baseline_score
FEATURE_WEIGHTS = {
    "synopsis": 0.45, "genre": 0.20, "type": 0.10,
    "studio": 0.05, "source": 0.05, "bayesian": 0.15,
}
WEIGHT_SUM = sum(FEATURE_WEIGHTS.values())

print(f"Static lookup tables ready  |  catalog={len(catalog):,}  embeddings={embeddings.shape}")

Static lookup tables ready  |  catalog=9,185  embeddings=(9185, 768)


In [9]:
# 2.3.b Process pair batches and write feature part files
RERANKER_FEATURES_DIR.mkdir(parents=True, exist_ok=True)
for old in RERANKER_FEATURES_DIR.glob("part_*.parquet"):
    old.unlink()

final_cols = [
    "user_id",
    "query_anime_id",
    "candidate_anime_id",
    "label",
    "candidate_source",
    # --- query-item features (all available at inference) ---
    "embedding_similarity",
    "genre_overlap",
    "type_match",
    "studio_match",
    "source_match",
    "candidate_avg_rating",
    "candidate_rating_count",
    "candidate_positive_ratio",
    "candidate_bayesian_norm",
    "candidate_popularity_rank",
    "rating_count_log",
    "score_diff",
    # --- new features ---
    "candidate_popularity_percentile",  # normalized popularity 0-1
    "embedding_rank",                   # position in embedding retrieval list
    "similarity_percentile",            # relative emb sim within query group
    "metadata_match_count",             # sum of type+studio+source matches
    "title_token_overlap",              # Jaccard over title tokens (franchise)
    "genre_overlap_count",              # raw genre intersection count
    "candidate_rating_cv",              # rating_std / avg_rating (polarization)
    "baseline_score",                   # Stage 1 multi-feature weighted score
]

part_count = 0
total_rows = 0

for pairs in iter_pair_batches(PAIR_BATCH_SIZE):
    pairs["user_id"] = pd.to_numeric(pairs["user_id"], errors="coerce")
    pairs["query_anime_id"] = pd.to_numeric(pairs["query_anime_id"], errors="coerce")
    pairs["candidate_anime_id"] = pd.to_numeric(pairs["candidate_anime_id"], errors="coerce")
    pairs = pairs.dropna(subset=["user_id", "query_anime_id", "candidate_anime_id"]).copy()
    if pairs.empty:
        continue

    pairs["user_id"] = pairs["user_id"].astype("int64")
    pairs["query_anime_id"] = pairs["query_anime_id"].astype("int64")
    pairs["candidate_anime_id"] = pairs["candidate_anime_id"].astype("int64")
    pairs["label"] = pd.to_numeric(pairs["label"], errors="coerce").fillna(0).astype("int8")

    features = pairs.copy()
    features["query_index"] = features["query_anime_id"].map(anime_to_index)
    features["candidate_index"] = features["candidate_anime_id"].map(anime_to_index)

    # --- Embedding similarity ---
    valid_idx = features["query_index"].notna() & features["candidate_index"].notna()
    features["embedding_similarity"] = 0.0
    if valid_idx.any():
        q_idx = features.loc[valid_idx, "query_index"].astype("int64").to_numpy()
        c_idx = features.loc[valid_idx, "candidate_index"].astype("int64").to_numpy()
        features.loc[valid_idx, "embedding_similarity"] = np.sum(
            embeddings[q_idx] * embeddings[c_idx], axis=1
        ).astype(np.float32)

    # --- Embedding rank: position in full retrieval list for each query ---
    features["embedding_rank"] = 0
    for q_aid, grp_idx in features.groupby("query_anime_id").groups.items():
        q_emb_idx = anime_to_index.get(int(q_aid))
        if q_emb_idx is None:
            continue
        # Compute similarities against ALL items for this query
        sims_all = embeddings @ embeddings[q_emb_idx]
        sims_all[q_emb_idx] = -1.0
        # Rank: higher sim = lower rank number (rank 1 = most similar)
        rank_order = np.argsort(-sims_all)
        rank_lookup = np.empty_like(rank_order)
        rank_lookup[rank_order] = np.arange(1, len(rank_order) + 1)

        row_idx = np.array(list(grp_idx), dtype=np.int64)
        cand_emb_idx = pd.to_numeric(
            features.loc[row_idx, "candidate_index"], errors="coerce"
        ).fillna(-1).to_numpy(dtype=np.int64)
        valid = (cand_emb_idx >= 0) & (cand_emb_idx < embeddings.shape[0])
        features.loc[row_idx[valid], "embedding_rank"] = rank_lookup[cand_emb_idx[valid]]

    # --- Similarity percentile: rank(emb_sim) within query group ---
    features["similarity_percentile"] = 0.0
    for _, grp_idx in features.groupby("query_anime_id").groups.items():
        row_idx = np.array(list(grp_idx), dtype=np.int64)
        sims = features.loc[row_idx, "embedding_similarity"].to_numpy(dtype=np.float64)
        n = len(sims)
        if n <= 1:
            features.loc[row_idx, "similarity_percentile"] = 1.0
        else:
            order = np.argsort(sims)
            ranks = np.empty_like(order, dtype=np.float64)
            ranks[order] = np.arange(n, dtype=np.float64)
            features.loc[row_idx, "similarity_percentile"] = (ranks / (n - 1)).astype(np.float32)

    # --- Genre / type / studio / source ---
    features["query_genres"] = features["query_anime_id"].map(genre_map)
    features["candidate_genres"] = features["candidate_anime_id"].map(genre_map)
    features["query_type"] = features["query_anime_id"].map(type_map)
    features["candidate_type"] = features["candidate_anime_id"].map(type_map)
    features["query_studio"] = features["query_anime_id"].map(studio_map)
    features["candidate_studio"] = features["candidate_anime_id"].map(studio_map)
    features["query_source"] = features["query_anime_id"].map(source_map)
    features["candidate_source_text"] = features["candidate_anime_id"].map(source_map)
    features["query_score"] = pd.to_numeric(features["query_anime_id"].map(score_map), errors="coerce")
    features["candidate_score"] = pd.to_numeric(features["candidate_anime_id"].map(score_map), errors="coerce")

    q_sets = features["query_genres"].map(_parse_genres_to_set)
    c_sets = features["candidate_genres"].map(_parse_genres_to_set)
    features["genre_overlap"] = [float(_jaccard(a, b)) for a, b in zip(q_sets, c_sets)]
    features["genre_overlap_count"] = [float(len(a & b)) for a, b in zip(q_sets, c_sets)]
    features["type_match"] = (features["query_type"] == features["candidate_type"]).astype("int8")
    features["studio_match"] = (features["query_studio"] == features["candidate_studio"]).astype("int8")
    features["source_match"] = (features["query_source"] == features["candidate_source_text"]).astype("int8")

    # --- metadata_match_count: sum of binary metadata matches ---
    features["metadata_match_count"] = (
        features["type_match"].astype("int8")
        + features["studio_match"].astype("int8")
        + features["source_match"].astype("int8")
    ).astype("int8")

    # --- Title token overlap (franchise detection) ---
    q_tokens = features["query_anime_id"].map(title_tokens_map)
    c_tokens = features["candidate_anime_id"].map(title_tokens_map)
    features["title_token_overlap"] = [
        float(_jaccard(qt if isinstance(qt, set) else set(),
                       ct if isinstance(ct, set) else set()))
        for qt, ct in zip(q_tokens, c_tokens)
    ]

    # --- Item quality features ---
    features["candidate_avg_rating"] = pd.to_numeric(
        features["candidate_anime_id"].map(candidate_avg_rating_map), errors="coerce"
    ).fillna(0.0)
    features["candidate_rating_count"] = pd.to_numeric(
        features["candidate_anime_id"].map(candidate_rating_count_map), errors="coerce"
    ).fillna(0.0)
    features["candidate_positive_ratio"] = pd.to_numeric(
        features["candidate_anime_id"].map(candidate_positive_ratio_map), errors="coerce"
    ).fillna(0.0)
    features["candidate_bayesian_norm"] = pd.to_numeric(
        features["candidate_anime_id"].map(candidate_bayesian_norm_map), errors="coerce"
    ).fillna(0.0)
    features["candidate_popularity_rank"] = pd.to_numeric(
        features["candidate_anime_id"].map(candidate_popularity_rank_map), errors="coerce"
    ).fillna(0.0)
    features["rating_count_log"] = np.log1p(features["candidate_rating_count"].astype(float))
    features["score_diff"] = (
        features["query_score"].fillna(0.0) - features["candidate_score"].fillna(0.0)
    ).abs()

    # --- candidate_popularity_percentile ---
    features["candidate_popularity_percentile"] = pd.to_numeric(
        features["candidate_anime_id"].map(candidate_popularity_percentile_map), errors="coerce"
    ).fillna(0.0)

    # --- candidate_rating_cv: coefficient of variation (polarization) ---
    cand_std = pd.to_numeric(
        features["candidate_anime_id"].map(candidate_rating_std_map), errors="coerce"
    ).fillna(0.0)
    cand_avg = features["candidate_avg_rating"].copy()
    features["candidate_rating_cv"] = np.where(
        cand_avg > 0.01, cand_std / cand_avg, 0.0
    ).astype(np.float32)

    # --- baseline_score: Stage 1 multi-feature weighted score ---
    features["baseline_score"] = (
        FEATURE_WEIGHTS["synopsis"] * features["embedding_similarity"]
        + FEATURE_WEIGHTS["genre"]   * features["genre_overlap"]
        + FEATURE_WEIGHTS["type"]    * features["type_match"].astype("float32")
        + FEATURE_WEIGHTS["studio"]  * features["studio_match"].astype("float32")
        + FEATURE_WEIGHTS["source"]  * features["source_match"].astype("float32")
        + FEATURE_WEIGHTS["bayesian"]* features["candidate_bayesian_norm"]
    ).astype("float32") / WEIGHT_SUM

    # --- Assemble output ---
    feature_df = features[final_cols].copy().fillna(0.0)
    feature_df["user_id"] = feature_df["user_id"].astype("int64")
    feature_df["query_anime_id"] = feature_df["query_anime_id"].astype("int64")
    feature_df["candidate_anime_id"] = feature_df["candidate_anime_id"].astype("int64")
    feature_df["label"] = feature_df["label"].astype("int8")
    feature_df["type_match"] = feature_df["type_match"].astype("int8")
    feature_df["studio_match"] = feature_df["studio_match"].astype("int8")
    feature_df["source_match"] = feature_df["source_match"].astype("int8")
    feature_df["metadata_match_count"] = feature_df["metadata_match_count"].astype("int8")

    float_cols = [
        "embedding_similarity", "genre_overlap", "candidate_avg_rating",
        "candidate_rating_count", "candidate_positive_ratio", "candidate_bayesian_norm",
        "candidate_popularity_rank", "rating_count_log", "score_diff",
        "candidate_popularity_percentile", "similarity_percentile",
        "title_token_overlap", "genre_overlap_count", "candidate_rating_cv",
        "baseline_score",
    ]
    for col in float_cols:
        feature_df[col] = pd.to_numeric(feature_df[col], errors="coerce").fillna(0.0).astype("float32")
    feature_df["embedding_rank"] = pd.to_numeric(
        feature_df["embedding_rank"], errors="coerce"
    ).fillna(0).astype("int32")

    out_path = RERANKER_FEATURES_DIR / f"part_{part_count:05d}.parquet"
    feature_df.to_parquet(out_path, index=False)

    total_rows += len(feature_df)
    part_count += 1

    del features, feature_df, pairs

print(f"Feature part files: {part_count}")
print(f"Total feature rows: {total_rows:,}")

Feature part files: 30
Total feature rows: 3,000,000


In [10]:
# 2.3.c Summarize chunked output
part_files = sorted(RERANKER_FEATURES_DIR.glob("part_*.parquet"))
if not part_files:
    raise RuntimeError("No feature part files were generated.")

part_rows = []
for part in part_files:
    df_part = pd.read_parquet(part, columns=["label"])
    part_rows.append(len(df_part))

print(f"Parts generated: {len(part_files)}")
print(f"Total rows across parts: {sum(part_rows):,}")
print(f"First part: {part_files[0]}")
print(f"Last part: {part_files[-1]}")

Parts generated: 30
Total rows across parts: 3,000,000
First part: ../../data/processed/reranker_features_parts/part_00000.parquet
Last part: ../../data/processed/reranker_features_parts/part_00029.parquet


In [11]:
# 2.3.d Optional preview (first part)
preview = pd.read_parquet(part_files[0]).head()
print(f"Preview columns: {list(preview.columns)}")
preview

Preview columns: ['user_id', 'query_anime_id', 'candidate_anime_id', 'label', 'candidate_source', 'embedding_similarity', 'genre_overlap', 'type_match', 'studio_match', 'source_match', 'candidate_avg_rating', 'candidate_rating_count', 'candidate_positive_ratio', 'candidate_bayesian_norm', 'candidate_popularity_rank', 'rating_count_log', 'score_diff', 'candidate_popularity_percentile', 'embedding_rank', 'similarity_percentile', 'metadata_match_count', 'title_token_overlap', 'genre_overlap_count', 'candidate_rating_cv', 'baseline_score']


,user_id,query_anime_id,candidate_anime_id,label,candidate_source,embedding_similarity,genre_overlap,type_match,studio_match,source_match,...,rating_count_log,score_diff,candidate_popularity_percentile,embedding_rank,similarity_percentile,metadata_match_count,title_token_overlap,genre_overlap_count,candidate_rating_cv,baseline_score
0,30,1535,16592,1,positive_history,0.415734,0.00,1,0,0,...,11.066701,1.41,0.982036,925,0.286667,1,0.0,0.0,0.232596,0.383795
1,30,1535,27899,0,hard_negative,0.480922,0.00,1,0,1,...,11.490721,1.60,0.994992,243,0.390000,2,0.0,0.0,0.297311,0.458202
2,30,1535,22319,0,hard_negative,0.463268,0.00,1,0,1,...,11.800387,0.83,0.999020,372,0.368889,2,0.0,0.0,0.216801,0.471809
3,30,1535,6707,0,hard_negative,0.382334,0.25,1,0,1,...,10.667885,1.48,0.960261,1562,0.226667,2,0.0,1.0,0.257995,0.466531
4,30,1535,1818,0,hard_negative,0.434923,0.00,1,1,1,...,10.946322,0.88,0.968209,663,0.317778,3,0.0,0.0,0.183118,0.505948


In [12]:
# 2.3.e Note
print("Chunked features are saved as parquet parts in:")
print(RERANKER_FEATURES_DIR)
print("Use these parts directly for training, or concatenate later if needed.")

Chunked features are saved as parquet parts in:
../../data/processed/reranker_features_parts
Use these parts directly for training, or concatenate later if needed.


In [13]:
# 2.4 Query groups + 2.5 user-based train/val/test split

TRAIN_PATH = DATA_ROOT / "processed" / "reranker_train.parquet"
VAL_PATH = DATA_ROOT / "processed" / "reranker_val.parquet"
TEST_PATH = DATA_ROOT / "processed" / "reranker_test.parquet"

part_files = sorted(RERANKER_FEATURES_DIR.glob("part_*.parquet"))
if not part_files:
    raise RuntimeError("No feature part files found. Run 2.3 first.")

# 1) Collect unique users from feature parts
all_users = set()
for part in part_files:
    u = pd.read_parquet(part, columns=["user_id"])["user_id"]
    u = pd.to_numeric(u, errors="coerce").dropna().astype("int64")
    all_users.update(u.unique().tolist())

all_users = np.array(sorted(all_users), dtype=np.int64)
rng = np.random.default_rng(42)
rng.shuffle(all_users)

n_users = len(all_users)
n_train = int(n_users * 0.70)
n_val = int(n_users * 0.15)
n_test = n_users - n_train - n_val

train_users = set(all_users[:n_train].tolist())
val_users = set(all_users[n_train:n_train + n_val].tolist())
test_users = set(all_users[n_train + n_val:].tolist())

print(f"Unique users: {n_users:,}")
print(f"Train users: {len(train_users):,} | Val users: {len(val_users):,} | Test users: {len(test_users):,}")

# 2) Stream parts and write split files
train_counts = {}
val_counts = {}
test_counts = {}

def _update_counts(counter: dict, frame: pd.DataFrame) -> None:
    vc = frame.groupby(["user_id", "query_anime_id"]).size()
    for (uid, qid), cnt in vc.items():
        key = (int(uid), int(qid))
        counter[key] = counter.get(key, 0) + int(cnt)

# Try streaming parquet writes with pyarrow; fallback to in-memory concat if unavailable
use_pyarrow = True
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception:
    use_pyarrow = False

for output in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if output.exists():
        output.unlink()

if use_pyarrow:
    train_writer = None
    val_writer = None
    test_writer = None
    try:
        for part in part_files:
            chunk = pd.read_parquet(part)
            chunk["user_id"] = pd.to_numeric(chunk["user_id"], errors="coerce")
            chunk = chunk.dropna(subset=["user_id"]).copy()
            chunk["user_id"] = chunk["user_id"].astype("int64")

            train_chunk = chunk[chunk["user_id"].isin(train_users)]
            val_chunk = chunk[chunk["user_id"].isin(val_users)]
            test_chunk = chunk[chunk["user_id"].isin(test_users)]

            if not train_chunk.empty:
                _update_counts(train_counts, train_chunk)
                t = pa.Table.from_pandas(train_chunk, preserve_index=False)
                if train_writer is None:
                    train_writer = pq.ParquetWriter(TRAIN_PATH, t.schema)
                train_writer.write_table(t)

            if not val_chunk.empty:
                _update_counts(val_counts, val_chunk)
                t = pa.Table.from_pandas(val_chunk, preserve_index=False)
                if val_writer is None:
                    val_writer = pq.ParquetWriter(VAL_PATH, t.schema)
                val_writer.write_table(t)

            if not test_chunk.empty:
                _update_counts(test_counts, test_chunk)
                t = pa.Table.from_pandas(test_chunk, preserve_index=False)
                if test_writer is None:
                    test_writer = pq.ParquetWriter(TEST_PATH, t.schema)
                test_writer.write_table(t)
    finally:
        if train_writer is not None:
            train_writer.close()
        if val_writer is not None:
            val_writer.close()
        if test_writer is not None:
            test_writer.close()
else:
    train_chunks = []
    val_chunks = []
    test_chunks = []
    for part in part_files:
        chunk = pd.read_parquet(part)
        chunk["user_id"] = pd.to_numeric(chunk["user_id"], errors="coerce")
        chunk = chunk.dropna(subset=["user_id"]).copy()
        chunk["user_id"] = chunk["user_id"].astype("int64")

        train_chunk = chunk[chunk["user_id"].isin(train_users)]
        val_chunk = chunk[chunk["user_id"].isin(val_users)]
        test_chunk = chunk[chunk["user_id"].isin(test_users)]

        if not train_chunk.empty:
            _update_counts(train_counts, train_chunk)
            train_chunks.append(train_chunk)
        if not val_chunk.empty:
            _update_counts(val_counts, val_chunk)
            val_chunks.append(val_chunk)
        if not test_chunk.empty:
            _update_counts(test_counts, test_chunk)
            test_chunks.append(test_chunk)

    if train_chunks:
        pd.concat(train_chunks, ignore_index=True).to_parquet(TRAIN_PATH, index=False)
    if val_chunks:
        pd.concat(val_chunks, ignore_index=True).to_parquet(VAL_PATH, index=False)
    if test_chunks:
        pd.concat(test_chunks, ignore_index=True).to_parquet(TEST_PATH, index=False)

# 3) LambdaRank query groups (sorted by user_id for deterministic grouping)
train_group_sizes = np.array([train_counts[uid] for uid in sorted(train_counts)], dtype=np.int64)
val_group_sizes = np.array([val_counts[uid] for uid in sorted(val_counts)], dtype=np.int64)
test_group_sizes = np.array([test_counts[uid] for uid in sorted(test_counts)], dtype=np.int64)

print(f"Saved: {TRAIN_PATH}")
print(f"Saved: {VAL_PATH}")
print(f"Saved: {TEST_PATH}")
print(f"Train groups: {len(train_group_sizes):,} | example: {train_group_sizes[:10]}")
print(f"Val groups: {len(val_group_sizes):,} | example: {val_group_sizes[:10]}")
print(f"Test groups: {len(test_group_sizes):,} | example: {test_group_sizes[:10]}")

# Optional quick row counts
def _safe_count_rows(path: Path) -> int:
    if not path.exists():
        return 0
    return len(pd.read_parquet(path, columns=["user_id"]))

print(f"Train rows: {_safe_count_rows(TRAIN_PATH):,}")
print(f"Val rows: {_safe_count_rows(VAL_PATH):,}")
print(f"Test rows: {_safe_count_rows(TEST_PATH):,}")

Unique users: 5,595
Train users: 3,916 | Val users: 839 | Test users: 840
Saved: ../../data/processed/reranker_train.parquet
Saved: ../../data/processed/reranker_val.parquet
Saved: ../../data/processed/reranker_test.parquet
Train groups: 104,905 | example: [20 22 20 20 17 22 25 22 21 20]
Val groups: 22,583 | example: [21 18 22 21 20 18 19 19 20 20]
Test groups: 22,836 | example: [18 17 21 19 19 16 17 19 19 18]
Train rows: 2,094,266
Val rows: 449,532
Test rows: 456,202


### Reranker Training

In [14]:
# 4.1 Train LambdaRank model
from pathlib import Path
import numpy as np
import pandas as pd

try:
    import lightgbm as lgb
except Exception as exc:
    raise ImportError(
        "lightgbm is required. Install it in this notebook environment (e.g., pip install lightgbm)."
    ) from exc

TRAIN_PATH = DATA_ROOT / "processed" / "reranker_train.parquet"
VAL_PATH = DATA_ROOT / "processed" / "reranker_val.parquet"
TEST_PATH = DATA_ROOT / "processed" / "reranker_test.parquet"

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing split file: {p}")

train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

required_cols = {"user_id", "label"}
for name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = required_cols - set(frame.columns)
    if missing:
        raise ValueError(f"{name} split missing required columns: {missing}")

# Sort by (user_id, query_anime_id) so each query forms one contiguous LambdaRank group
train_df = train_df.sort_values(["user_id", "query_anime_id"]).reset_index(drop=True)
val_df = val_df.sort_values(["user_id", "query_anime_id"]).reset_index(drop=True)
test_df = test_df.sort_values(["user_id", "query_anime_id"]).reset_index(drop=True)

group_train = train_df.groupby(["user_id", "query_anime_id"]).size().to_numpy(dtype=np.int64)
group_val = val_df.groupby(["user_id", "query_anime_id"]).size().to_numpy(dtype=np.int64)
group_test = test_df.groupby(["user_id", "query_anime_id"]).size().to_numpy(dtype=np.int64)

# Exclude ID columns, label, and the diagnostic string column candidate_source
exclude_cols = {
    "label", "user_id", "anime_id",
    "query_anime_id", "candidate_anime_id",
    "candidate_source",  # string diagnostic column, not a feature
}
candidate_feature_cols = [c for c in train_df.columns if c not in exclude_cols]

numeric_feature_cols = [c for c in candidate_feature_cols if pd.api.types.is_numeric_dtype(train_df[c])]
dropped_non_numeric = [c for c in candidate_feature_cols if c not in numeric_feature_cols]

feature_cols = numeric_feature_cols
if not feature_cols:
    raise ValueError("No numeric feature columns found after exclusions.")

X_train = train_df[feature_cols].fillna(0.0).to_numpy(dtype=np.float32)
y_train = train_df["label"].to_numpy(dtype=np.int32)
X_val = val_df[feature_cols].fillna(0.0).to_numpy(dtype=np.float32)
y_val = val_df["label"].to_numpy(dtype=np.int32)
X_test = test_df[feature_cols].fillna(0.0).to_numpy(dtype=np.float32)
y_test = test_df["label"].to_numpy(dtype=np.int32)

train_data = lgb.Dataset(
    X_train,
    label=y_train,
    group=group_train,
    feature_name=feature_cols,
    free_raw_data=False,
    )

val_data = lgb.Dataset(
    X_val,
    label=y_val,
    group=group_val,
    reference=train_data,
    feature_name=feature_cols,
    free_raw_data=False,
    )

params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [5, 10, 20],
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[val_data],
    valid_names=["val"],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)

print(f"Train rows: {len(train_df):,} | Val rows: {len(val_df):,} | Test rows: {len(test_df):,}")
print(f"Train groups: {len(group_train):,} | Val groups: {len(group_val):,} | Test groups: {len(group_test):,}")
print(f"Features used ({len(feature_cols)}): {feature_cols[:12]}{' ...' if len(feature_cols) > 12 else ''}")
if dropped_non_numeric:
    print(f"Dropped non-numeric features ({len(dropped_non_numeric)}): {dropped_non_numeric[:12]}{' ...' if len(dropped_non_numeric) > 12 else ''}")

Training until validation scores don't improve for 50 rounds
[50]	val's ndcg@5: 0.564833	val's ndcg@10: 0.624144	val's ndcg@20: 0.631229
Early stopping, best iteration is:
[1]	val's ndcg@5: 0.596725	val's ndcg@10: 0.652964	val's ndcg@20: 0.65942
Train rows: 2,094,266 | Val rows: 449,532 | Test rows: 456,202
Train groups: 104,905 | Val groups: 22,583 | Test groups: 22,836
Features used (20): ['embedding_similarity', 'genre_overlap', 'type_match', 'studio_match', 'source_match', 'candidate_avg_rating', 'candidate_rating_count', 'candidate_positive_ratio', 'candidate_bayesian_norm', 'candidate_popularity_rank', 'rating_count_log', 'score_diff'] ...


In [15]:
# 4.2 Evaluation metrics: NDCG@K, MAP@10, HitRate@10
from collections import defaultdict

def _group_index_ranges(group_sizes: np.ndarray):
    start = 0
    for g in group_sizes:
        end = start + int(g)
        yield start, end
        start = end

def _dcg_at_k(labels_sorted: np.ndarray, k: int) -> float:
    labels = labels_sorted[:k]
    if labels.size == 0:
        return 0.0
    gains = (2 ** labels - 1).astype(np.float64)
    discounts = np.log2(np.arange(2, labels.size + 2))
    return float(np.sum(gains / discounts))

def _ndcg_at_k(y_true: np.ndarray, y_pred: np.ndarray, group_sizes: np.ndarray, k: int) -> float:
    scores = []
    for s, e in _group_index_ranges(group_sizes):
        labels = y_true[s:e]
        preds = y_pred[s:e]
        if labels.size == 0:
            continue
        order_pred = np.argsort(-preds)
        order_ideal = np.argsort(-labels)
        dcg = _dcg_at_k(labels[order_pred], k)
        idcg = _dcg_at_k(labels[order_ideal], k)
        scores.append(0.0 if idcg == 0 else dcg / idcg)
    return float(np.mean(scores)) if scores else 0.0

def _map_at_k(y_true: np.ndarray, y_pred: np.ndarray, group_sizes: np.ndarray, k: int) -> float:
    ap_scores = []
    for s, e in _group_index_ranges(group_sizes):
        labels = y_true[s:e]
        preds = y_pred[s:e]
        if labels.size == 0:
            continue
        order = np.argsort(-preds)[:k]
        ranked = labels[order]
        pos = 0
        precision_sum = 0.0
        for i, rel in enumerate(ranked, start=1):
            if rel > 0:
                pos += 1
                precision_sum += pos / i
        ap_scores.append(0.0 if pos == 0 else precision_sum / pos)
    return float(np.mean(ap_scores)) if ap_scores else 0.0

def _hit_rate_at_k(y_true: np.ndarray, y_pred: np.ndarray, group_sizes: np.ndarray, k: int) -> float:
    hits = []
    for s, e in _group_index_ranges(group_sizes):
        labels = y_true[s:e]
        preds = y_pred[s:e]
        if labels.size == 0:
            continue
        order = np.argsort(-preds)[:k]
        hits.append(1.0 if np.any(labels[order] > 0) else 0.0)
    return float(np.mean(hits)) if hits else 0.0

def evaluate_ranking(y_true: np.ndarray, y_pred: np.ndarray, group_sizes: np.ndarray) -> dict:
    return {
        "NDCG@10": _ndcg_at_k(y_true, y_pred, group_sizes, 10),
        "NDCG@20": _ndcg_at_k(y_true, y_pred, group_sizes, 20),
        "MAP@10": _map_at_k(y_true, y_pred, group_sizes, 10),
        "HitRate@10": _hit_rate_at_k(y_true, y_pred, group_sizes, 10),
    }

val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

val_metrics = evaluate_ranking(y_val, val_pred, group_val)
test_metrics = evaluate_ranking(y_test, test_pred, group_test)

print("Validation metrics:")
for k, v in val_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")

metric_targets = {
    "NDCG@10": 0.70,
    "NDCG@20": 0.65,
    "MAP@10": 0.50,
    "HitRate@10": 0.80,
}

print("\nAgainst targets (test):")
for metric, target in metric_targets.items():
    value = test_metrics[metric]
    status = "PASS" if value >= target else "MISS"
    print(f"  {metric}: {value:.4f} (target>{target:.2f}) [{status}]")

Validation metrics:
  NDCG@10: 0.6475
  NDCG@20: 0.6551
  MAP@10: 0.5467
  HitRate@10: 0.9714

Test metrics:
  NDCG@10: 0.6443
  NDCG@20: 0.6524
  MAP@10: 0.5432
  HitRate@10: 0.9696

Against targets (test):
  NDCG@10: 0.6443 (target>0.70) [MISS]
  NDCG@20: 0.6524 (target>0.65) [PASS]
  MAP@10: 0.5432 (target>0.50) [PASS]
  HitRate@10: 0.9696 (target>0.80) [PASS]


In [16]:
# 4.2.b Sanity checks for suspiciously perfect metrics
def _group_positive_counts(df: pd.DataFrame) -> pd.Series:
    return df.groupby("user_id")["label"].sum().astype(int)

def _group_sizes(df: pd.DataFrame) -> pd.Series:
    return df.groupby("user_id").size().astype(int)

for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    self_mask = (df["query_anime_id"] == df["candidate_anime_id"]) if {"query_anime_id", "candidate_anime_id"}.issubset(df.columns) else pd.Series(False, index=df.index)
    pos_counts = _group_positive_counts(df)
    size_counts = _group_sizes(df)
    print(f"[{split_name}] rows={len(df):,} users={df['user_id'].nunique():,}")
    print(f"  self-candidate rows: {int(self_mask.sum()):,} ({self_mask.mean()*100:.2f}%)")
    print(f"  avg candidates/user: {size_counts.mean():.2f} | min={size_counts.min()} | max={size_counts.max()}")
    print(f"  users with >=1 positive: {(pos_counts > 0).mean()*100:.2f}%")
    print(f"  positives/user: mean={pos_counts.mean():.2f} | min={pos_counts.min()} | max={pos_counts.max()}")

[train] rows=2,094,266 users=3,916
  self-candidate rows: 0 (0.00%)
  avg candidates/user: 534.80 | min=32 | max=670
  users with >=1 positive: 100.00%
  positives/user: mean=26.79 | min=2 | max=30
[val] rows=449,532 users=839
  self-candidate rows: 0 (0.00%)
  avg candidates/user: 535.79 | min=32 | max=657
  users with >=1 positive: 100.00%
  positives/user: mean=26.92 | min=2 | max=30
[test] rows=456,202 users=840
  self-candidate rows: 0 (0.00%)
  avg candidates/user: 543.10 | min=32 | max=664
  users with >=1 positive: 100.00%
  positives/user: mean=27.19 | min=2 | max=30


In [17]:
# 4.2.c Adjusted evaluation excluding trivial self-candidates
if {"query_anime_id", "candidate_anime_id"}.issubset(test_df.columns):
    test_eval_df = test_df[test_df["query_anime_id"] != test_df["candidate_anime_id"]].copy()
else:
    test_eval_df = test_df.copy()

if test_eval_df.empty:
    print("Adjusted test set is empty after removing self-candidates.")
else:
    test_eval_df = test_eval_df.sort_values(["user_id", "query_anime_id"]).reset_index(drop=True)
    group_test_adj = test_eval_df.groupby(["user_id", "query_anime_id"]).size().to_numpy(dtype=np.int64)
    X_test_adj = test_eval_df[feature_cols].fillna(0.0).to_numpy(dtype=np.float32)
    y_test_adj = test_eval_df["label"].to_numpy(dtype=np.int32)
    test_pred_adj = model.predict(X_test_adj)
    test_metrics_adj = evaluate_ranking(y_test_adj, test_pred_adj, group_test_adj)

    print(f"Adjusted test rows: {len(test_eval_df):,} (removed {len(test_df) - len(test_eval_df):,})")
    print("Adjusted test metrics (no self-candidates):")
    for k, v in test_metrics_adj.items():
        print(f"  {k}: {v:.4f}")

Adjusted test rows: 456,202 (removed 0)
Adjusted test metrics (no self-candidates):
  NDCG@10: 0.6443
  NDCG@20: 0.6524
  MAP@10: 0.5432
  HitRate@10: 0.9696


In [18]:
# 4.3 Baseline comparison — Stage 1 multi-feature score as baseline
possible_baseline_cols = [
    "baseline_score",
    "multi_feature_score",
    "content_score",
    "score",
]
baseline_col = next((c for c in possible_baseline_cols if c in test_df.columns), None)

if baseline_col is None:
    print("No baseline score column found in test split. Skipping baseline comparison.")
    print("  available columns:", sorted(test_df.columns.tolist()))
else:
    baseline_metrics = evaluate_ranking(
        y_test, test_df[baseline_col].to_numpy(dtype=np.float32), group_test
    )
    print(f"Using baseline column: '{baseline_col}'")
    print(f"  (this is the Stage 1 weighted multi-feature score from recommendation_v1.py)")
    print("\nBaseline vs Reranker (test):")
    for metric in ["NDCG@10", "NDCG@20", "MAP@10", "HitRate@10"]:
        b = baseline_metrics[metric]
        r = test_metrics[metric]
        rel = ((r - b) / b * 100.0) if b > 0 else np.nan
        rel_str = f"{rel:+.2f}%" if np.isfinite(rel) else "n/a"
        print(f"  {metric}: baseline={b:.4f} | reranker={r:.4f} | relative={rel_str}")

    ndcg10_rel = (
        (test_metrics["NDCG@10"] - baseline_metrics["NDCG@10"])
        / baseline_metrics["NDCG@10"] * 100.0
    ) if baseline_metrics["NDCG@10"] > 0 else np.nan
    if np.isfinite(ndcg10_rel):
        print(f"\nNDCG@10 relative lift: {ndcg10_rel:+.2f}%")
        print("Deployment criterion (>= +5%):", "PASS" if ndcg10_rel >= 5.0 else "MISS")

Using baseline column: 'baseline_score'
  (this is the Stage 1 weighted multi-feature score from recommendation_v1.py)

Baseline vs Reranker (test):
  NDCG@10: baseline=0.0941 | reranker=0.6443 | relative=+584.68%
  NDCG@20: baseline=0.2667 | reranker=0.6524 | relative=+144.64%
  MAP@10: baseline=0.0536 | reranker=0.5432 | relative=+914.21%
  HitRate@10: baseline=0.2354 | reranker=0.9696 | relative=+311.93%

NDCG@10 relative lift: +584.68%
Deployment criterion (>= +5%): PASS


In [19]:
# 4.5 Optional hyperparameter tuning scaffold (manual grid)
search_space = {
    "num_leaves": [15, 31, 63],
    "learning_rate": [0.01, 0.05, 0.1],
    "min_data_in_leaf": [10, 20, 50],
}

print("Manual tuning space:")
for k, v in search_space.items():
    print(f"  {k}: {v}")
print("Tip: keep early stopping enabled; compare by validation NDCG@10.")

Manual tuning space:
  num_leaves: [15, 31, 63]
  learning_rate: [0.01, 0.05, 0.1]
  min_data_in_leaf: [10, 20, 50]
Tip: keep early stopping enabled; compare by validation NDCG@10.


In [20]:
# 4.4 Feature importance analysis
importance_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance_gain": model.feature_importance(importance_type="gain"),
        "importance_split": model.feature_importance(importance_type="split"),
    }
).sort_values("importance_gain", ascending=False).reset_index(drop=True)

print("Top 20 features by gain:")
display(importance_df.head(20))

popularity_like = importance_df[importance_df["feature"].str.contains("popularity|rating_count|bayesian|score", case=False, na=False)]
print(f"Popularity-like features in top 20: {sum(popularity_like.index < 20)}")

Top 20 features by gain:


,feature,importance_gain,importance_split
0,embedding_rank,227180.158966,4
1,candidate_avg_rating,36243.308472,7
2,candidate_rating_count,14640.487823,8
3,candidate_positive_ratio,5241.434875,3
4,candidate_popularity_rank,5117.129822,2
5,similarity_percentile,1990.824051,4
6,studio_match,885.856995,1
7,metadata_match_count,405.057007,1
8,type_match,0.000000,0
9,genre_overlap,0.000000,0


Popularity-like features in top 20: 7


In [21]:
# 4.6 Save model + metadata
RERANKER_DIR = ARTIFACT_DIR / "reranker_v2"
RERANKER_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = RERANKER_DIR / "model.txt"
FEATURES_PATH = RERANKER_DIR / "feature_columns.json"
IMPORTANCE_PATH = RERANKER_DIR / "feature_importance.csv"

model.save_model(str(MODEL_PATH))
importance_df.to_csv(IMPORTANCE_PATH, index=False)

import json
with open(FEATURES_PATH, "w", encoding="utf-8") as f:
    json.dump({"feature_columns": feature_cols}, f, ensure_ascii=False, indent=2)

print(f"Saved model: {MODEL_PATH}")
print(f"Saved feature columns: {FEATURES_PATH}")
print(f"Saved feature importance: {IMPORTANCE_PATH}")

Saved model: ../../ml/artifacts/content_based_anime_v1/reranker_v2/model.txt
Saved feature columns: ../../ml/artifacts/content_based_anime_v1/reranker_v2/feature_columns.json
Saved feature importance: ../../ml/artifacts/content_based_anime_v1/reranker_v2/feature_importance.csv


## Phase 4 — Model Training & Evaluation

### 4.1 LightGBM LambdaRank
Train a ranking model with query groups defined by `user_id`.

### 4.2 Evaluation metrics
Compute `NDCG@10`, `NDCG@20`, `MAP@10`, and `HitRate@10` on validation/test splits.

### 4.3 Baseline comparison
Compare reranker metrics with baseline scores if a baseline score column exists.

### 4.4 Feature importance
Inspect model feature importance and check for popularity-heavy behavior.

### 4.5 Hyperparameter tuning
Optional search scaffold (manual grid / Optuna-ready structure).

### 4.6 Save model
Persist model artifact to `ml/artifacts/reranker_v1/model.txt`.